# The drum machine as a class
## Lab 05 · ACTAM 2026

In the last lesson a drum machine was built in the browser, in the style of the Roland TR-808: sixteen keys, a LED on each, a step that moves along. The code was split in three parts:

- the **model**: the data that describes the machine, `keys` (sixteen `true`/`false`) and `step`;
- the **controller**: the functions that change the model, such as a click that flips `keys[i]` or `nextStep`;
- the **view**: the function that draws the model, `render`.

Today we build the same machine in Python, with a **class**. A class keeps the model (its attributes) and the controller (its methods) together in one object. The views are ways of looking at that object: as text, as coloured keys, as sound.

### Setup

Run this cell first. It has `play`, three drum sounds in numpy, `mix` and `bar_html`, which draws a row of keys. Nothing in it is an exercise: read it if you are curious.

## A · The model

### A1 · A pattern from a string
**Core · Write a class**

In lesson 04 a pattern was a string: `"x"` for a hit, `"."` for a rest. Now it becomes an object.

Write a class `Pattern` whose constructor takes such a string and keeps the model in one attribute, `steps`: a list of booleans, `True` for a hit. Any other symbol raises a `ValueError` with a message of your own.

```python
kick_p = Pattern("x...x...x...x...")
kick_p.steps        # [True, False, False, False, True, ...]
```

### A2 · Text, length and steps
**Core · Add methods**

Evaluate `kick_p` alone in a cell: Python shows something like `<__main__.Pattern at 0x10a2f...>`. That tells us nothing.

Add four methods:

- `text()` gives back the string, `"x...x..."`;
- `__repr__` uses it to show `Pattern('x...x...')`;
- `__len__` makes `len(kick_p)` work: the number of steps;
- `__getitem__` makes `kick_p[i]` work: `True` if step `i` is a hit.

Each section writes the class again with all the methods so far, and the new ones are left for you. In a file you would keep one class and add to it; in a notebook, writing it again lets every section run on its own. After writing a class again, make its objects again: the old ones still belong to the old class.

## B · Two more views

### B1 · Keys on the screen
**Core · Add a method**

When the last value of a cell is an object with a method called `_repr_html_`, Jupyter calls it and shows the HTML it gives back, instead of the text of `__repr__`. It is the same idea as `__repr__`: a name that someone else calls for us.

The setup has `bar_html(steps)`, which makes the HTML of a row of keys from a list of booleans. Add `_repr_html_` to `Pattern`: one line.

The object is the same; we look at it in two ways. Neither view is in the model: the colours of the keys live in `bar_html`, not in `steps`. Change `COLOURS` and every pattern looks different, with the same model.

### B2 · The sound view
**Core · Write a function**

A third view is sound. Write `render(pattern, sound, step_s=0.125)`, a function outside the class:

1. a step lasts `n = round(step_s * SR)` samples;
2. start from `np.zeros(len(pattern) * n + len(sound))`: the extra samples let the last sound ring;
3. for every step `i` that is a hit, add `sound` into the slice that starts at sample `i * n`.

At `step_s=0.125`, sixteen steps last two seconds: a bar of sixteenth notes at 120 BPM.

`render` uses only `len(pattern)` and `pattern[i]`: it works on any object that has those two methods. It does not need to know how `Pattern` stores its steps.

### B3 · A beat
**Core · Try**

Three patterns, three sounds, one `mix`. The beat below is the one of *Billie Jean*, from lesson 04. Change it into a beat of your own.

## C · The controller

### C1 · Change the model
**Core · Add methods**

The controller changes the model. Add three methods:

- `toggle(i)` flips step `i`: what a click on a key did in the browser;
- `clear()` turns every step off;
- `mirror()` reverses the pattern, so that the last step comes first.

For `mirror`, a slice with step `-1` reads a list backwards: `[1, 2, 3][::-1]` is `[3, 2, 1]`.

These methods change `self.steps` and give back nothing.

Since `mirror` gives back nothing, `q = p.mirror()` makes `q` equal to `None`, and `p` is the one that changes. The operators of lesson 07 worked the other way: `a >> b` made a new clip and left `a` and `b` as they were. Both are good designs; a method should make clear which one it follows.

## D · Equal, or the same?

### D1 · Save and load
In the browser, saving was `store = keys.slice()`. With `store = keys` alone, saving did not work. Here is the same thing in Python.

`store = p` does not copy anything. It gives a second name to the same object, as in JavaScript. `is` asks whether two names refer to the same object.

Here is a first attempt at a copy. Before running the next cell, guess: does `new` change when `p` changes?

`new` is a different object, but its `steps` is the same list as the one of `p`: one list with two names, one level down.

### D2 · A real copy, and equality
**Core · Add methods**

Add two methods:

- `copy()` gives back a new pattern with the same steps and a list of its own. There is a short way: `text()` and the constructor.
- `__eq__(self, other)` makes `==` compare the steps. Without it, `==` behaves like `is`.

Load copies again: with `p = store`, the next `toggle` would change what was saved. With a copy, save, load and reset are each one line, because the model is all in `steps`.

## E · Operators

### E1 · Then, again, together
**Core · Add methods**

As with clips in lesson 07, but on patterns:

| expression | meaning | method |
|---|---|---|
| `a >> b` | `a`, then `b` | `__rshift__` |
| `a * 3` | `a` three times | `__mul__` |
| `a \| b` | a hit where either has one | `__or__` |

All three give back a new pattern and leave `a` and `b` as they were. `text()` makes the first two short. For `|`, raise a `ValueError` if the two patterns have different lengths.

`basic * 3 >> fill` is `(basic * 3) >> fill`: `*` comes before `>>`, as in lesson 07.

### E2 · Four bars
**Core · Try**

Write four bars of a beat with operators: three bars of a groove and one with a fill. Play them.

## F · Save the state, not the machine

### F1 · A pattern as JSON
**Core · Add methods**

The model can be saved as text, for a file, a database or a message, and read back to rebuild the same machine. A common format is JSON, and the `json` module turns lists, dictionaries, numbers and booleans into text and back:

```python
json.dumps({"steps": [True, False]})      # '{"steps": [true, false]}'
json.loads('{"steps": [true, false]}')    # {'steps': [True, False]}
```

Add two methods:

- `to_json()` gives back the JSON text of `{"steps": ...}`;
- `from_json(text)` builds a new pattern from such a text. It is called on the class, `Pattern.from_json(text)`, not on a pattern: we do not have one yet.

A method called on the class is marked with `@classmethod`. Its first argument is the class itself, called `cls`, so `cls("")` makes a new, empty pattern. It is another way to build an object.

Only the steps are saved: no colours, no sounds, no tempo. Anyone with this text and the class can rebuild the same pattern.

**Try:** copy the printed text into `Pattern.from_json('...')`, change one `false` into `true`, and look at the keys.

## G · Extensions

### G1 · A drum machine of patterns
**Extension · Write a class**

A drum machine holds a pattern per instrument, a tempo and the current step: the model of the last lesson, `keys` and `step`, for three tracks. An object that holds other objects is called **composition**.

Write `DrumMachine(tracks, step_s=0.125)`, where `tracks` is a dictionary from a name in `kit` to a `Pattern`. It keeps `tracks`, `step_s` and `step = 0`, and has:

- `length`, a property: the number of steps of the longest track;
- `tick()`, the controller of the step: one step on, back to 0 after the last;
- `render()`, the sound view: every track rendered with its sound, mixed;
- `_repr_html_()`, a row of keys per track, with the current step marked: `bar_html(steps, step)` draws its border green.

Change a step with `dm.tracks["snare"].toggle(15)` and look at `dm` again: the machine shows the change, because it holds the pattern itself, not a copy.

### G2 · Euclidean rhythms
**Extension · Write a function**

Spread `k` hits as evenly as possible over `n` steps: step `i` is a hit when `(i * k) % n < k`. With 3 hits over 8 steps this gives `x..x..x.`, the *tresillo* of Cuban music, also heard in much pop music. Many traditional rhythms come out of this rule.

Write `euclid(k, n)`, which gives back a `Pattern`. Then try `euclid(5, 8)`, `euclid(5, 16)` and `euclid(7, 12)`.

### G3 · Three against four
**Extension · Try**

Twelve steps hold three groups of four and four groups of three. Play `Pattern("x...") * 3` on one sound and `Pattern("x..") * 4` on another, together. Which one sounds like the beat?

### G4 · Your own keys
**Extension · Write a function**

Write `my_bar_html(steps)`, your own version of `bar_html`: a `<div>` per step with inline CSS, as in the last lesson. Other colours, other shapes, a different LED. Then try it with `display(HTML(my_bar_html(kick_p.steps)))`.

### G5 · A dataclass
**Extension · Try**

`@dataclass`, from lesson 07, writes `__init__`, `__repr__` and `__eq__` for us from the list of fields. Here is a pattern as a dataclass, with no `__eq__` written by hand. Do two such patterns with the same steps compare equal? Which methods of our `Pattern` could a dataclass not write for us?